In [ ]:
# Install the required packages:
# - langchain: core LangChain framework
# - langchain-openai: LangChain's integration with OpenAI models (GPT series)
!pip install -q langchain langchain-openai


In [ ]:
from google.colab import userdata

# LangChain message types:
# AIMessage    — a response generated by the AI model
# HumanMessage — a message from the user
# SystemMessage — a standing instruction that shapes the model's behavior (not shown to the end user)
from langchain.messages import AIMessage, HumanMessage, SystemMessage

# Helpers for building multi-modal message content blocks (text + images)
from langchain_core.messages.content import create_image_block, create_text_block

# ChatOpenAI: LangChain's wrapper around OpenAI's chat models (e.g., GPT-4, o1)
from langchain_openai import ChatOpenAI

# BaseModel: Pydantic base class used to define structured output schemas
from pydantic import BaseModel, SecretStr

# Securely load the OpenAI API key from Colab secrets
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper to pretty-print a model response with token usage statistics.
# Monitoring tokens is important because API costs are billed per token.
def print_response(response: AIMessage):
    print(f"Response id: {response.id}")
    if response.usage_metadata is not None:
        input_tokens = response.usage_metadata.get("input_tokens", 0)
        # Cache-read tokens were served from OpenAI's prompt cache (cheaper / faster)
        cached_tokens = response.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
        output_tokens = response.usage_metadata.get("output_tokens", 0)
        # Reasoning tokens are consumed internally by thinking models (e.g., o1) before final output
        reasoning_tokens = response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        print(f"Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)")

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)


## Basic usage

Note that LangChain's OpenAI integration uses the `Chat Completions API` by default. This behavior can be controlled through the `use_responses_api` parameter (but keep in mind that the output format may change).

In [ ]:
# Create a default GPT model instance.
# By default, LangChain uses the Chat Completions API endpoint.
openai_default_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key)

# Send a two-message conversation:
# - SystemMessage sets the model's role/persona
# - HumanMessage is the actual user request
museum_audio_guide_response = openai_default_model.invoke(
    input=[
        SystemMessage("You are a helpful history teaching assistant."),
        HumanMessage("Write a short museum audio-guide introduction for first-time visitors standing in front of the Rosetta Stone. Keep it under 5 sentences.")
    ]
)


In [ ]:
# Print the museum audio guide response with token usage stats
print_response(museum_audio_guide_response)


In [ ]:
# Demonstrate using the newer Responses API instead of the default Chat Completions API.
# use_responses_api=True switches to OpenAI's stateful Responses API endpoint.
# Note: the output format/fields may differ slightly from Chat Completions.
openai_responses_api_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, use_responses_api=True)

documentary_script_response = openai_responses_api_model.invoke(
    input=[
        SystemMessage("You are a professional scriptwriter."),
        HumanMessage("Draft a cinematic 6-line opening voice-over for a documentary about bioluminescent oceans.")
    ]
)


In [ ]:
# Print the documentary script response from the Responses API model
print_response(documentary_script_response)


## Streaming

In [ ]:
# Demonstrate streaming — receive and print the model's response token by token.
# Streaming is useful for interactive UIs where you want to show text as it is generated
# rather than waiting for the complete response.
for chunk in openai_default_model.stream(
    input=[
        SystemMessage("You are an expert in culinary."),
        HumanMessage("Design a one-evening street-food route through Seoul for a curious first-time visitor who wants bold flavors but no seafood.")
    ]
):
    if chunk.text:
        # end="" avoids extra newlines; flush=True sends each character to the terminal immediately
        print(chunk.text, end="", flush=True)


## Multi-turn conversations

In [ ]:
# Demonstrate multi-turn conversation — the model remembers details across turns
# because the entire conversation history is passed with each request.
conversation = [
    SystemMessage("You are concise and remember user details from the chat history you receive."),
    HumanMessage("My name is Maria. I live in Plovdiv and I am preparing for a Python exam."),
]

# Turn 1: send opening message and receive first reply
first_reply = openai_default_model.invoke(conversation)
# Append the AI reply so the model has the full context on the next turn
conversation.append(first_reply)


In [ ]:
# Print the model's first reply
print_response(first_reply)


In [ ]:
# Turn 2: Follow-up question — because we pass the full history, the model remembers Maria's details
conversation.append(
    HumanMessage("What do you remember about me, and what should I focus on this week?")
)

second_reply = openai_default_model.invoke(conversation)
conversation.append(second_reply)


In [ ]:
# Print the second reply — the model should recall Maria's name, city, and exam context
print_response(second_reply)


## Reasoning

In [ ]:
# Use a high-reasoning model for a creative writing task.
# reasoning_effort="high" tells the model to spend more time reasoning before responding,
# which improves quality for complex or nuanced tasks (at the cost of more tokens and latency).
openai_high_reasoning_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="high")

book_cover_response = openai_high_reasoning_model.invoke(
    input=[
        HumanMessage("Write a back-cover blurb for a literary novel about a family-run cinema trying to survive in the streaming era.")
    ]
)


In [ ]:
# Print the reasoning-enhanced book cover blurb — token stats show reasoning tokens consumed
print_response(book_cover_response)


## Structured output

In [ ]:
# Define a Pydantic schema for the structured output.
# Pydantic models specify exactly what fields the model's response must contain.
# with_structured_output() instructs the model to return JSON matching this schema,
# which LangChain then validates and parses into a Python object automatically.
class WorkshopBrief(BaseModel):
    title: str                      # Workshop title
    audience: str                   # Who the workshop is intended for
    duration_minutes: float         # Total duration in minutes
    key_takeaways: list[str]        # List of main learning outcomes
    materials_needed: list[str]     # Items participants should bring or prepare

# Configure the model to always return a WorkshopBrief object instead of free-form text
openai_structured_output_model = openai_default_model.with_structured_output(WorkshopBrief)

workshop_brief = openai_structured_output_model.invoke(
    input=[
        SystemMessage("You are an expert event organizer."),
        HumanMessage("Design a beginner-friendly Saturday workshop about balcony herb gardening.")
    ]
)


In [ ]:
# Display the parsed WorkshopBrief object.
# This is a typed Python object — not raw text — so its fields can be accessed programmatically.
workshop_brief


## Vision

<img src="https://freerangestock.com/sample/88947/painter-working-in-studio.jpg" />

In [ ]:
# Demonstrate vision (multi-modal) capability — send both text and an image to GPT.
# OpenAI's vision-enabled models can analyze images alongside text in the same request.
# create_text_block(): wraps a plain text string as a message content block
# create_image_block(url=...): references an image by URL for the model to analyze
analyze_image_response = openai_default_model.invoke(
    input=[
        SystemMessage("You are an expert image analyst. Keep your answer concise and structured."),
        HumanMessage(
            content_blocks=[
              # Tell the model what to do with the image
              create_text_block("Analyze this image and return a one-sentence summary followed by 5 key visible objects."),
              # The image the model will visually analyze
              create_image_block(url="https://freerangestock.com/sample/88947/painter-working-in-studio.jpg")
            ]
        )
    ]
)


In [ ]:
# Print the image analysis result — a summary sentence and 5 visible objects from the painting image
print_response(analyze_image_response)
